In [62]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import glob
from scipy import linalg
from scipy.spatial.distance import cdist
import os
from PIL import Image
import math
import numpy as np

In [66]:
def detectTag(folder_path, marker_size): 
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    aruco_params = cv2.aruco.DetectorParameters()
    aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

    all_object_pts = []
    all_img_pts = []
    marker_size_in_world = np.array([
        [0,0,0], 
        [marker_size, 0, 0], 
        [marker_size, marker_size, 0],
        [0, marker_size, 0]], dtype=np.float32)
    image_size = None

    image_paths = glob.glob(os.path.join(folder_path, "*.jpg"))
    print(f"Found {len(image_paths)} images")


    
    for image_path in image_paths: 
        image = cv2.imread(image_path)
        #print(f"Original image size: {(image.shape[1], image.shape[0])}")
        if image is None: 
            continue

        target_height = 300 
        height, width = image.shape[:2]
        if height > target_height:
            scale = target_height / height
            new_width = int(width * scale)
            new_height = target_height
            image = cv2.resize(image, (new_width, new_height))
        
        if image_size is None: 
            image_size = (image.shape[1], image.shape[0]) 

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        corners, ids, _ = aruco_detector.detectMarkers(gray)
        
        if ids is not None:
            for marker_corners in corners: 
                corners_2d = marker_corners.reshape(-1, 2)
                all_object_pts.append(marker_size_in_world)
                all_img_pts.append(corners_2d)
        else:
            print(f"No markers detected in {os.path.basename(image_path)}")

    print(f"Total markers collected: {len(all_object_pts)}")
    
    print("Starting calibration...")
    
    error, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        all_object_pts,  
        all_img_pts,  
        image_size,     
        None, None,
        flags=cv2.CALIB_USE_LU  
    )
    
    return camera_matrix, dist_coeffs, error

In [67]:
folder_path = "./0.1/converted_jpgs"
marker_size = 0.05
camera_matrix, dist_coeffs, error = detectTag(folder_path, marker_size)
print("Camera Matrix:\n", camera_matrix)
print("Distortion Coefficients:\n", dist_coeffs)
print("Reprojection Error:", error)


Found 44 images
Total markers collected: 244
Starting calibration...
Camera Matrix:
 [[304.2629568    0.         195.16980775]
 [  0.         309.21555782 138.67387668]
 [  0.           0.           1.        ]]
Distortion Coefficients:
 [[ 3.87692841e-01 -2.68804917e+00 -1.14291298e-02  2.50477764e-03
   5.22306467e+00]]
Reprojection Error: 0.36274058519782104


In [46]:
import sys
print(sys.executable)

c:\Users\isabe\cs\cs180\IHU3025.github.io\.venv\Scripts\python.exe


In [68]:
import viser
import numpy as np

In [ ]:
# def estimating_camera(folder_path, marker_size, camera_matrix, dist_coeffs): 
#     aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
#     aruco_params = cv2.aruco.DetectorParameters()
#     aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

#     all_matrix = []
#     all_image = []
#     all_size = []
#     #intrisitc
   
#     marker_size_in_world = np.array([
#         [0,0,0], 
#         [marker_size, 0, 0], 
#         [marker_size, marker_size, 0],
#         [0, marker_size, 0]], dtype=np.float32)
    

#     image_paths = glob.glob(os.path.join(folder_path, "*.jpg"))
#     print(f"Found {len(image_paths)} images")

#     for image_path in image_paths: 
#         print(f"processing -- {image_path}")
#         image = cv2.imread(image_path)
#         if image is None: 
#             continue

#         target_height = 200 
#         height, width = image.shape[:2]
#         if height > target_height:
#             scale = target_height / height
#             new_width = int(width * scale)
#             new_height = target_height
#             image_resize = cv2.resize(image, (new_width, new_height))
        
        
#         image_resize_size = (image_resize.shape[1], image_resize.shape[0]) 
#         image_resize_rbg = cv2.cvtColor(image_resize, cv2.COLOR_BGR2RGB)

#         gray = cv2.cvtColor(image_resize, cv2.COLOR_BGR2GRAY)

#         corners, ids, _ = aruco_detector.detectMarkers(gray)
        
#         if ids is not None:
#             this_obj_pts = []
#             this_img_pts = [] 
#             for marker_corners in corners: 
#                 corners_2d = marker_corners.reshape(-1, 2)
#                 this_obj_pts.append(marker_size_in_world)
#                 this_img_pts.append(corners_2d)
#             success, rvec, tvec = cv2.solvePnP(
#                 np.array(this_obj_pts).reshape(-1, 1, 3), 
#                 np.array(this_img_pts).reshape(-1, 1, 2),
#                 camera_matrix, dist_coeffs)
#             if success: 
#                 R, _ = cv2.Rodrigues(rvec)
#                 Rt = np.hstack((R, tvec)) 
#                 w2c = np.vstack((Rt, np.array([0, 0, 0, 1])))
#                 c2w = np.linalg.inv(w2c)
#                 all_matrix.append(c2w)
#                 all_image.append(image_resize_rbg)
#                 all_size.append(image_resize_size)
#         else:
#             print(f"No markers detected in {os.path.basename(image_path)}")
#     return all_image, all_matrix, all_size
    

In [74]:
def estimating_camera(folder_path, marker_size, camera_matrix, dist_coeffs): 
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    aruco_params = cv2.aruco.DetectorParameters()
    aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, aruco_params)

    all_matrix = []
    all_image = []
    all_size = []
    
    # Define the tag corners in world coordinates (same as reference code)
    tag_size = marker_size
    s = tag_size / 2
    marker_size_in_world = np.array([
        [-s,  s, 0],   # TL
        [ s,  s, 0],   # TR  
        [ s, -s, 0],   # BR
        [-s, -s, 0],   # BL
    ], dtype=np.float32)

    image_paths = glob.glob(os.path.join(folder_path, "*.jpg"))
    print(f"Found {len(image_paths)} images")

    for image_path in image_paths: 
        print(f"processing -- {image_path}")
        image = cv2.imread(image_path)
        if image is None: 
            continue

        # Resize (keeping your resize logic)
        target_height = 300 
        height, width = image.shape[:2]
        if height > target_height:
            scale = target_height / height
            new_width = int(width * scale)
            new_height = target_height
            image_resize = cv2.resize(image, (new_width, new_height))
        else:
            image_resize = image
        
        image_resize_size = (image_resize.shape[1], image_resize.shape[0]) 
        image_resize_rbg = cv2.cvtColor(image_resize, cv2.COLOR_BGR2RGB)

        gray = cv2.cvtColor(image_resize, cv2.COLOR_BGR2GRAY)

        corners, ids, _ = aruco_detector.detectMarkers(gray)
        
        if ids is not None:
            target_id = 0  # Using the same target ID as reference code
            flat_ids = ids.flatten()

            if target_id not in flat_ids:
                print(f"[WARN] Tag {target_id} not found in {os.path.basename(image_path)}, skipped.")
                continue

            # Find the index of our target tag
            idx = np.where(flat_ids == target_id)[0][0]
            imgp = corners[idx].reshape(-1, 2).astype(np.float32)

            # --- Use the reference code's PnP approach ---
            ret = cv2.solvePnPGeneric(
                marker_size_in_world, imgp, camera_matrix, dist_coeffs,
                flags=cv2.SOLVEPNP_IPPE_SQUARE  # Using IPPE for better stability
            )

            success = ret[0]
            rvecs = ret[1]
            tvecs = ret[2]

            if not success or len(rvecs) < 2:
                print(f"[WARN] PnP failed for {os.path.basename(image_path)}")
                continue

            # Get both solutions (main and ambiguous)
            R1, _ = cv2.Rodrigues(rvecs[0])
            t1 = tvecs[0].reshape(3)

            R2, _ = cv2.Rodrigues(rvecs[1])  
            t2 = tvecs[1].reshape(3)

            # Calculate camera positions in world coordinates
            cam1 = -R1.T @ t1  # Camera position for solution 1
            cam2 = -R2.T @ t2  # Camera position for solution 2

            # Choose the solution where camera is above the tag (z > 0)
            if cam1[2] > cam2[2]:
                R, t = R1, t1
            else:
                R, t = R2, t2

            # Construct c2w matrix (same as reference code)
            c2w = np.eye(4, dtype=np.float32)
            c2w[:3, :3] = R.T
            c2w[:3, 3] = -R.T @ t

            all_matrix.append(c2w)
            all_image.append(image_resize_rbg)
            all_size.append(image_resize_size)
            
            print(f"[INFO] Pose OK for {os.path.basename(image_path)}, t = {t.ravel()}")

        else:
            print(f"No markers detected in {os.path.basename(image_path)}")
    
    print(f"[INFO] {len(all_matrix)} valid poses found.")
    return all_image, all_matrix, all_size

In [77]:
import time
def display_camera(all_image, all_matrix, all_size, K):
    server = viser.ViserServer(share=True)
    # Example of visualizing a camera frustum (in practice loop over all images)
    # c2w is the camera-to-world transformation matrix (3x4), and K is the camera intrinsic matrix (3x3)
    for i in range(len(all_image)): 
        H = all_size[i][1]
        W = all_size[i][0]
        c2w = all_matrix[i]
        img = all_image[i]
        server.scene.add_camera_frustum(
            f"/cameras/{i}", # give it a name
            fov=2 * np.arctan2(H / 2, K[0, 0]), # field of view
            aspect=W / H, # aspect ratio
            scale=0.01, # scale of the camera frustum change if too small/big
            wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz, # orientation in quaternion format
            position=c2w[:3, 3], # position of the camera
            image=img # image to visualize
        )

    while True:
        time.sleep(0.1)  # Wait to allow visualization to run

In [78]:
folder_path = "./G2/converted_jpgs"
all_image, all_matrix, all_size = estimating_camera(folder_path, 0.05, camera_matrix, dist_coeffs)
display_camera(all_image, all_matrix, all_size, camera_matrix)

Found 140 images
processing -- ./G2/converted_jpgs\IMG_9295.jpg
No markers detected in IMG_9295.jpg
processing -- ./G2/converted_jpgs\IMG_9296.jpg
No markers detected in IMG_9296.jpg
processing -- ./G2/converted_jpgs\IMG_9297.jpg
[INFO] Pose OK for IMG_9297.jpg, t = [-0.03029353  0.04582248  0.20581884]
processing -- ./G2/converted_jpgs\IMG_9298.jpg
[INFO] Pose OK for IMG_9298.jpg, t = [-0.03661178  0.05861899  0.19076171]
processing -- ./G2/converted_jpgs\IMG_9299.jpg
[INFO] Pose OK for IMG_9299.jpg, t = [-0.0393542   0.05585547  0.18842861]
processing -- ./G2/converted_jpgs\IMG_9300.jpg
[INFO] Pose OK for IMG_9300.jpg, t = [-0.06147415  0.07487383  0.21812602]
processing -- ./G2/converted_jpgs\IMG_9301.jpg
[INFO] Pose OK for IMG_9301.jpg, t = [-0.06283741  0.06478717  0.23182499]
processing -- ./G2/converted_jpgs\IMG_9302.jpg
[INFO] Pose OK for IMG_9302.jpg, t = [-0.09419969  0.06294874  0.23855761]
processing -- ./G2/converted_jpgs\IMG_9303.jpg
[INFO] Pose OK for IMG_9303.jpg, t = [

(viser) Connection closed (0, 0 total)

[INFO] Pose OK for IMG_9339.jpg, t = [-0.02776614  0.05901322  0.22783496]
processing -- ./G2/converted_jpgs\IMG_9340.jpg
[INFO] Pose OK for IMG_9340.jpg, t = [-0.02674608  0.05135139  0.22739848]
processing -- ./G2/converted_jpgs\IMG_9341.jpg
[INFO] Pose OK for IMG_9341.jpg, t = [-0.03719482  0.02416523  0.24327523]
processing -- ./G2/converted_jpgs\IMG_9342.jpg
[INFO] Pose OK for IMG_9342.jpg, t = [-0.01986105  0.01862455  0.2385042 ]
processing -- ./G2/converted_jpgs\IMG_9343.jpg
No markers detected in IMG_9343.jpg
processing -- ./G2/converted_jpgs\IMG_9344.jpg
[INFO] Pose OK for IMG_9344.jpg, t = [-0.0878408  -0.00071049  0.27753915]
processing -- ./G2/converted_jpgs\IMG_9345.jpg
[INFO] Pose OK for IMG_9345.jpg, t = [-0.05284513 -0.00940908  0.27346468]
processing -- ./G2/converted_jpgs\IMG_9346.jpg
[INFO] Pose OK for IMG_9346.jpg, t = [-0.07025825 -0.01683132  0.25492066]
processing -- ./G2/converted_jpgs\IMG_9347.jpg
[INFO] Pose OK for IMG_9347.jpg, t = [-0.07362241 -0.02192476  

╭────── viser (listening *:8097) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8097   │
│   Websocket │ ws://localhost:8097     │
│             ╵                         │
╰───────────────────────────────────────╯

(viser) Share URL requested!

(viser) Generated share URL (expires in 24 hours, max 16 clients): https://bagged-keyframe.share.viser.studio

(viser) Connection opened (0, 1 total), 466 persistent messages

(viser) Connection closed (0, 0 total)

KeyboardInterrupt: 

In [79]:
from sklearn.model_selection import train_test_split

def process_dataset(all_image, all_matrix, all_size, camera_matrix, dist_coeffs, 
                    output_path = 'my+data.npz'): 
    # Undistort an image using the calibration results
    undistorted_imgs = []
    for image in all_image: 
        #undistorted_img = cv2.undistort(image, camera_matrix, dist_coeffs)
        #undistorted_imgs.append(undistorted_img)
        undistorted_imgs.append(image)

    image_array = np.array(undistorted_imgs) #(N, H, W , 3)
    c2w_array = np.array(all_matrix)    #(N, 4, 4)

    n_total = len(image_array)
    train_val_idx, test_idx = train_test_split(range(n_total), 
                                           test_size = 0.1, random_state= 42, shuffle= True)
    train_idx, val_idx = train_test_split(train_val_idx, 
                                          test_size = 0.11, random_state= 123, shuffle = True)
    
    #camera_matrix[[fx 0 cx] [0 fy cy] [0 0 1]]

    images_val = image_array[val_idx]
    images_train = image_array[train_idx]

    c2w_val = c2w_array[val_idx]
    c2ws_train = c2w_array[train_idx]
    c2w_test = c2w_array[test_idx]

    np.savez(
    output_path,
    images_train=images_train,    # (N_train, H, W, 3)
    c2ws_train=c2ws_train,        # (N_train, 4, 4)
    images_val=images_val,        # (N_val, H, W, 3)
    c2ws_val=c2w_val,            # (N_val, 4, 4)
    c2ws_test=c2w_test,          # (N_test, 4, 4)
    camera_matrix=camera_matrix
    )
    print(f"Dataset saved to {output_path}")
    

In [80]:
process_dataset(all_image, all_matrix, all_size, camera_matrix, dist_coeffs, 'g2_newCamerat1.npz')

Dataset saved to g2_newCamerat1.npz


In [37]:
data = np.load('lafufu_200undistort_originalCamera.npz')

# List all arrays stored in the file
print("Keys in the .npz file:")
print(data.files)

# Inspect each array
for key in data.files:
    print(f"\nKey: {key}")
    print(f"Shape: {data[key].shape}")
    print(f"Dtype: {data[key].dtype}")
    print(f"Preview:\n{data[key]}")


Keys in the .npz file:
['images_train', 'c2ws_train', 'images_val', 'c2ws_val', 'c2ws_test', 'camera_matrix']

Key: images_train
Shape: (25, 200, 266, 3)
Dtype: uint8
Preview:
[[[[ 96  71  44]
   [ 97  71  44]
   [ 95  69  42]
   ...
   [182 205 219]
   [190 216 228]
   [191 219 233]]

  [[ 95  68  44]
   [ 97  71  44]
   [ 97  70  43]
   ...
   [186 212 223]
   [191 218 229]
   [190 218 230]]

  [[ 94  68  41]
   [ 92  66  41]
   [ 97  71  44]
   ...
   [187 211 223]
   [188 215 227]
   [191 219 231]]

  ...

  [[145 116  84]
   [138 112  79]
   [138 112  79]
   ...
   [  4   5   7]
   [  3   5   7]
   [  4   5   9]]

  [[145 119  86]
   [147 121  86]
   [150 124  89]
   ...
   [  3   5   9]
   [  5   6  10]
   [  4   5   9]]

  [[154 128  93]
   [155 129  93]
   [155 129  94]
   ...
   [  3   4   6]
   [  3   4   6]
   [  9  10  14]]]


 [[[ 80 113 154]
   [ 78 103 141]
   [ 71  94 126]
   ...
   [ 63  43  19]
   [ 60  41  16]
   [ 62  42  18]]

  [[ 88 126 173]
   [ 78 110 154]
   [

In [39]:
data = np.load('lafufu_dataset - Copy.npz')

# List all arrays stored in the file
print("Keys in the .npz file:")
print(data.files)

# Inspect each array
for key in data.files:
    print(f"\nKey: {key}")
    print(f"Shape: {data[key].shape}")
    print(f"Dtype: {data[key].dtype}")
    print(f"Preview:\n{data[key]}")


Keys in the .npz file:
['images_train', 'c2ws_train', 'images_val', 'c2ws_val', 'K']

Key: images_train
Shape: (26, 214, 285, 3)
Dtype: uint8
Preview:
[[[[170 147 109]
   [165 143 105]
   [153 128  94]
   ...
   [155 131  96]
   [153 130  96]
   [153 130  95]]

  [[172 150 111]
   [169 146 108]
   [153 128  93]
   ...
   [158 134  98]
   [155 131  95]
   [160 137  99]]

  [[171 147 110]
   [170 147 109]
   [157 133  98]
   ...
   [155 130  95]
   [156 131  94]
   [158 133  95]]

  ...

  [[164 138  93]
   [153 125  82]
   [164 138  93]
   ...
   [123 103  72]
   [133 112  77]
   [129 108  76]]

  [[164 138  94]
   [156 128  85]
   [160 134  92]
   ...
   [128 107  76]
   [127 106  73]
   [134 112  79]]

  [[164 137  94]
   [159 131  86]
   [155 127  86]
   ...
   [132 111  79]
   [125 104  72]
   [126 105  73]]]


 [[[  3   7  17]
   [  5   9  20]
   [  5   9  19]
   ...
   [ 55  47  35]
   [ 49  41  28]
   [ 50  42  29]]

  [[  3   7  17]
   [  4   8  17]
   [  4   8  17]
   ...
   [ 